In [75]:
import subprocess
from dataclasses import dataclass, field

class Prop:
    def __and__(self, other):
        return And([self, other])
    def __or__(self, other):
        return Or([self, other])
    def __invert__(self):
        return Not(self)
    def __eq__(self, other):
        return Iff(self, other)

class Term:
    def __eq__(self, other):
        return Eq(self, other)
    def __ne__(self, other):
        return Not(Eq(self, other))
    def __add__(self, other):
        return App("add", [self, other])


@dataclass(frozen=True, eq=False)
class App(Term):
    name: str
    args: list[Term]
    def __repr__(self):
        if self.args:
            return f"{self.name}({','.join(map(repr, self.args))})"
        else:
            return self.name
        
@dataclass(frozen=True)
class FuncDecl:
    name : str
    arity : int
    def __call__(self, *args):
        if len(args) != self.arity:
            raise ValueError(f"Function {self.name} expects {self.arity} arguments, got {len(args)}")
        return App(self.name, list(args))

@dataclass(frozen=True, eq=False)
class Var(Term):
    name: str
    prop : Prop | None = None
    def __post_init__(self):
        if not self.name.isupper():
            raise ValueError("Variable names must be uppercase", self.name)
    def __repr__(self):
        return self.name

@dataclass(frozen=True, eq=False)
class PApp(Prop):
    name: str
    args: list[Term]
    def __repr__(self):
        if self.args:
            return f"{self.name}({','.join(map(repr, self.args))})"
        else:
            return self.name

@dataclass(frozen=True, eq=False)
class Eq(Prop):
    left: Term
    right: Term
    def __repr__(self):
        return f"({self.left} = {self.right})"


@dataclass(frozen=True, eq=False)
class And(Prop):
    children: list[Term]
    def __repr__(self):
        return f"({' & '.join(map(str, self.children))})"

@dataclass(frozen=True, eq=False)
class Or(Prop):
    children: list[Term]
    def __repr__(self):
        return f"({' | '.join(map(str, self.children))})"
    
@dataclass(frozen=True, eq=False)
class Not(Prop):
    child: Term
    def __repr__(self):
        return f"~({repr(self.child)})"

@dataclass(frozen=True, eq=False)
class Implies(Prop):
    hyp: Term
    conc: Term
    def __repr__(self):
        return f"({self.hyp} => {self.conc})"

@dataclass(frozen=True, eq=False)
class Iff(Prop):
    left: Term
    right: Term
    def __repr__(self):
        return f"({self.left} <=> {self.right})"

@dataclass(frozen=True, eq=False)
class ForAll(Prop):
    vars: list[Var]
    body: object
    def __repr__(self):
        return f"![{",".join(map(repr, self.vars))}]: ({self.body})"

# Could make ForAll just have these?
def QForAll(vars: list[Var], *body):
    if len(body) == 1:
        return ForAll(vars, body[0])
    elif len(body) == 2:
        return ForAll(vars, Implies(body[0], body[1]))
    else:
        return ForAll(vars, Implies(And(list(body[:-1])), body[-1]))

@dataclass(frozen=True, eq=False)
class Exists(Prop):
    vars: list[Var]
    body: object
    def __repr__(self):
        return f"?[{",".join(map(repr, self.vars))}]: ({self.body})"

true = App("true", [])
false = App("false", [])


@dataclass(frozen=True)
class Proof:
    # ctx
    # hyps
    goal: Prop
    by: list[object] = field(default_factory=list)
    def __repr__(self):
        return f"|- {self.goal}"
    def __and__(self, other): # and intro
        assert isinstance(other, Proof), f"Expected Proof, got {other}"
        return Proof(And([self.goal, other.goal]), self.by + other.by)
    def __or__(self, other): # or intro 1
        assert isinstance(other, Prop)
        return Proof(Or([self.goal, other]), self.by + other.by)
    def __call__(self, *args): # forall and implies elim
        if isinstance(self.goal, ForAll):
            ...
        elif isinstance(self.goal, Implies):
            assert isinstance(args[0], Proof), f"Expected Proof, got {args[0]}"
            assert args[0].goal == self.goal.hyp, f"Expected proof of {self.goal.hyp}, got {args[0].goal}"
            return Proof(self.goal.conc)(*args[1:])
    
            
def axiom(p : Prop) -> Proof:
    return Proof(p, ["axiom"])

def prove(p : Prop, by=[]) -> Proof:
    with open("/tmp/prob.p", "w") as f:
        for i, b in enumerate(by):
            assert isinstance(b, Proof), f"Expected Proof, got {b}"
            f.write(f"fof(ax, axiom, {b.goal}).\n")
        f.write(f"fof(goal, conjecture, {p}).\n")
    res = subprocess.run(["./nanocopi-ht.sh /tmp/prob.p 1"], shell=True, cwd="/home/philip/Downloads/nanoCoP-i-HT/", capture_output=True, text=True).stdout
    if "is a intu Theorem" in res:
        return Proof(p, by)
    else:
        raise ValueError(f"Failed to prove {p} with {by}. Result: {res}")

def Consts(names: str) -> list[App]:
    return [App(name, []) for name in names.split()]
def Vars(names: str) -> list[Var]:
    return [Var(name) for name in names.split()]
def Function(name, arity):
    def res(*args):
        if len(args) != arity:
            raise ValueError(f"Function {name} expects {arity} arguments, got {len(args)}")
        return App(name, list(args))
    return res
def Relation(name, arity):
    def res(*args):
        if len(args) != arity:
            raise ValueError(f"Relation {name} expects {arity} arguments, got {len(args)}")
        return PApp(name, list(args))
    return res
def Props(names: str) -> list[PApp]:
    return [PApp(name, []) for name in names.split()]
p,q,r = Props("p q r")
prove(Implies(p,p))

#prove(p | ~p)
elem = Relation("elem", 2)
emp = App("emp", [])
X,Y,Z = Vars("X Y Z")

ext = axiom(ForAll([X,Y], ForAll([Z], elem(Z, X) == elem(Z, Y)) == (X==Y)))

elem_emp = axiom(ForAll([X], ~(elem(X, emp))))

prove(~elem(emp, emp), by=[elem_emp])

upair = Function("upair", 2)
elem_upair = axiom(QForAll([X,Y,Z], elem(X, upair(Y,Z)) ==
                                    ((X==Y) | (X==Z))))

#_1 = prove(ForAll([X,Y], (elem() == (upair(X,Y) == upair(Y,X)) ), by=[ext, elem_upair])
prove(ForAll([X,Y], upair(X,Y) == upair(Y,X)), by=[ext, elem_upair])

defined = set()
def define(name, args, body):
    #assert name not in defined, f"{name} already defined"
    # Check for occurrence in body
    if isinstance(body, Prop):
        f = Relation(name, len(args))
    elif isinstance(body, Term):
        f = Function(name, len(args))
    return f, axiom(ForAll(args, f(*args) == body))
#upair(X,Y) == upair(X,Y)
sing, sing_defn = define("sing", [X], upair(X,X))
#elem_upair
elem_sing = prove(QForAll([X, Y], elem(X, sing(Y)) == (X==Y)), by=[ext, elem_upair, sing_defn])
elem_sing


pair = define("pair", [X,Y], upair(sing(X), upair(X,Y)))






# Peano


# Ring Kock Lawvere

https://pi.math.cornell.edu/~oconnor/sia.pdf

In [ ]:
x,y,z = Vars("X Y Z")

zero, one = Consts("zero one")

add_comm = axiom(ForAll([x,y], x + y == y + x))
add_assoc = axiom(ForAll([x,y,z], (x + y) + z == x + (y + z)))
add_zero = axiom(ForAll([x], x + zero == x))

zero_add = prove(ForAll([x], zero + x == x), by=[add_comm, add_zero])

mul_comm = axiom(ForAll([x,y], x * y == y * x))
mul_assoc = axiom(ForAll([x,y,z], (x * y) * z == x * (y * z)))
one_mul = axiom(ForAll([x], x * one == x))
mul_one = prove(ForAll([x], one * x == x), by=[mul_comm, one_mul])



#eps, = Consts("eps")
inf = Relation("inf", 1)
inf_defn = axiom(ForAll([x], inf(x) == (x * x == 0)))

def inf(x):
    return x*x == 0




TypeError: unsupported operand type(s) for *: 'Var' and 'Var'

In [ ]:
def kock(f):
    d,a = Vars("d a")
    return axiom(ForAll([d], d*d == 0, Exists([a], f(d) == f(0) + a*d)))

apply = Function("apply", 2)
def comp(f): # functional comprehension. Lambda lifting?
    F,x = Vars("F X")
    axiom(Exists[F], ForAll([x], apply(F,x) == f(x)))

fun = Relation("fun", 1)
def define2(name, args, body):
    F = Const(name)
    return axiom(And(fun(F), ForAll([args], apply(F, *args) == body)))
    
def ext

In [ ]:
@dataclass(frozen=True, eq=False)
class App(Term):
    name: str
    args: list[Term]
    def __repr__(self):
        if self.args:
            return f"{self.name}({','.join(map(repr, self.args))})"
        else:
            return self.name
        
    @classmethod
    def Var(cls, name):


    @classmethod
    def Const(cl, name):
        return cls(name, [])
    @classmethod

    

class Set(App):

class Real(App):
    def __add__(self, other):
        return Real("add", [self, other])
    
Real.Var("foo")


class Term:
    f : str
    args : list[Term] | None
    infix = False

    def is_var(self):
        return self.args is None
    
    @classmethod
    def Var(cls, name):
        return cls(name, None)
    @classmethod
    def Const(cls, name):
        return cls(name, [])
    @classmethod
    def App(cls, name, args):
        return cls(name, args)
    
class Prop(Term): ...
class Set(Term): ...     
class Real(Term):
    def __add__(self, other):
        return Real("add", [self, other])

Partial logic.
Carrying extra props in tags

In [ ]:
def qprove(vs, goal, by=[]): ...


class ProofZipper:
    vs : list[Var]
    hyps : list[Var]
    trail : list[ProofCtx]
    goal : Prop

@dataclass
class Intro:
    vs : list[Var]

@dataclass
class Exists:
    ...


class AVar(): ...
class EVar(): ...

type Ctx = list[AVar | EVar]



In [ ]:
class App:
    f : str
    args : list[App]
    infix = False

class Prop:
    f : str
    args : list[App | Prop]
    infix = False
    precedence = 0
    def tptp(self): ...

class ForAll: ...
class Exists: ...



In [61]:
prove(ForAll([X,Y], upair(X,Y) == upair(X,Y)))


ValueError: Failed to prove ![X,Y]: (True) with []. Result: Timeout


Proof in context makes more sense if you have Term/Prop in ctx also.

CZF
IZF

Synthetic Differentials

Category stuff 

Arith goals
Congruence closure

Locale
internal language of topoi
computable functions?
apply(F, X) 

Temporal modal?



In [ ]:
class Poly:
    f : str
    terms : dict[tuple[object, int], float]
    lift : list[bool]
    def __add__(self, other):
        if not isinstance(other, Poly):
            raise ValueError(f"Cannot add {other} to Poly")
        res = Poly()
        res.terms = self.terms.copy()
        for k,v in other.terms.items():
            if k in res.terms:
                res.terms[k] += v
            else:
                res.terms[k] = v
        return res
